# Решающие деревья

#  I. Немного решающих деревьев
##### Задача 1. Построение "среднего" алгоритма
В этом задании вам нужно построить графики, демонстрирующие, как алгоритм аппроксимирует истинную зависимость в данных и как он меняется в зависимости от гиперпараметров метода обучения.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.tree import DecisionTreeRegressor
%matplotlib inline

In [ ]:
def f(x):
    return np.sin(x)   # истинная зависимость в данных
sample_size = 100      # длина выборки
samples_num = 20       # количество выборок
linspace = np.linspace(0, 7, 1000)  # точки для построения графиков

__1. (2 балла)__
1. Сгенерируйте выборку $x$ из одномерного экспоненциального распределения (np.random.exponential) длины sample_size.
1. Создайте вектор целевых переменных $y$ как сумму $f(x)$ и случайного шума, сгенерированного из равномерного распределения на отрезке $[-1, 1]$ (np.random.uniform).
1. Обучите DecisionTreeRegressor с параметрами по умолчанию на полученной выборке и сделайте предсказания для объектов из linspace.
1. Постройте два графика на одном рисунке: $f(x)$ и зависимость, восстановленную решающим деревом.

    Рекомендация: не забудьте, что все методы обучения в sklearn требуют на вход двумерную матрицу объекты-признаки. Сделать такую из одномерного вектора можно добавлением мнимых осей (np.newaxis).

In [ ]:
x = np.random.exponential(scale=1.0, size=sample_size)
y = f(x) + np.random.uniform(-1, 1, size=sample_size)

tree = DecisionTreeRegressor()
tree.fit(x[:, np.newaxis], y)
y_pred = tree.predict(linspace[:, np.newaxis])

plt.figure(figsize=(10, 6))
plt.plot(linspace, f(linspace), 'r-', label='Истинная зависимость f(x)', linewidth=2)
plt.plot(linspace, y_pred, 'b-', label='Восстановленная зависимость', linewidth=2)
plt.scatter(x, y, alpha=0.5, s=20, label='Обучающая выборка', color='gray')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Аппроксимация функции решающим деревом')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

__2. (1 балл)__

Повторите первые 3 шага, описанные выше, samples_num раз. На одном графике для каждого обученного решающего дерева визуализируйте восстановленную им зависимость (рекомендуется все такие линии рисовать полупрозрачными и серым цветом: plt.plot(...... color="gray", alpha=0.5)).  На этом же графике изобразите истинную зависимость f(x) (красным цветом: color="red") и усредненную по всем деревьям восстановленную зависимость (черным цветом: color="black").    

In [ ]:
def depth(max_depth=None):
    predictions = []
    
    for i in range(samples_num):
        x = np.random.exponential(scale=1.0, size=sample_size)
        y = f(x) + np.random.uniform(-1, 1, size=sample_size)
        
        tree = DecisionTreeRegressor(max_depth=max_depth)
        tree.fit(x[:, np.newaxis], y)
        y_pred = tree.predict(linspace[:, np.newaxis])
        predictions.append(y_pred)
    
    predictions = np.array(predictions)
    avg_prediction = np.mean(predictions, axis=0)
    
    plt.figure(figsize=(12, 8))
    
    for pred in predictions:
        plt.plot(linspace, pred, color='gray', alpha=0.5, linewidth=0.5)
    
    plt.plot(linspace, f(linspace), 'r-', label='Истинная зависимость f(x)', linewidth=2)
    plt.plot(linspace, avg_prediction, 'k-', label='Усредненная зависимость', linewidth=2)
    
    plt.xlabel('x')
    plt.ylabel('y')
    depth_str = f' (max_depth={max_depth})' if max_depth is not None else ' (без ограничения глубины)'
    plt.title(f'Восстановление зависимости решающими деревьями{depth_str}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

depth(None)

__3.(0.5 балл):__
Повторите предыдущий пункт, установив максимальную глубину решающего дерева равной 2, а затем равной 4. Таким образом, у вас получится еще два графика.

In [ ]:
depth(2)
depth(4)

__4. (0.5 балла)__ Что можно сказать о смещении решающих деревьев, исходя из проведенного эксперимента? В каких из трех рассмотренных случаев (без ограничения на глубину дерева и с ограничением 2 и 4) можно утверждать, что смещение решающего дерева близко к нулю?

**Ответ:**

Смещение (bias) решающих деревьев можно оценить, сравнив усредненное предсказание с истинной функцией f(x).

1. **Без ограничения на глубину**: Смещение близко к нулю. Деревья могут переобучаться на конкретных выборках (высокая дисперсия), но в среднем они хорошо аппроксимируют истинную зависимость, так как могут построить достаточно сложную модель.

2. **С max_depth=2**: Смещение больше нуля. Деревья слишком простые и не могут достаточно точно аппроксимировать истинную зависимость (недообучение). Усредненное предсказание будет заметно отличаться от f(x).

3. **С max_depth=4**: Смещение меньше, чем при max_depth=2, но может быть больше, чем без ограничения. Деревья имеют среднюю сложность, что может давать компромисс между смещением и дисперсией.

Таким образом, смещение близко к нулю в случае без ограничения на глубину дерева, так как деревья могут построить достаточно сложную модель для аппроксимации истинной зависимости.

### II. Решающие деревья чужими руками

#### Задача 3.
В этой части вам нужно посмотреть на класс написанный за вас для обучения решающего дерева в задаче бинарной классификации с возможностью обработки вещественных и категориальных признаков.

__8. (1 балл)__

Загрузите таблицу [students.csv](https://drive.google.com/file/d/0B2zoFVYw1rN3a0d0Zm43TzQ4aUU/view?usp=sharing) (это немного преобразованный датасет [User Knowledge](https://archive.ics.uci.edu/ml/datasets/User+Knowledge+Modeling)). В ней признаки объекта записаны в первых пяти столбцах, а в последнем записана целевая переменная (класс: 0 или 1). Постройте на одном изображении пять кривых "порог — значение критерия Джини" для всех пяти признаков. Отдельно визуализируйте scatter-графики "значение признака — класс" для всех пяти признаков.

In [ ]:
import pandas as pd
data = pd.read_csv('students.csv')
data.head()

In [ ]:
from importlib import reload
from matplotlib import pyplot as plt
import hw3code
reload(hw3code)

In [ ]:
X = data.iloc[:, :5].values
y = data.iloc[:, 5].values

plt.figure(figsize=(15, 8))
feature_names = data.columns[:5].tolist()

for i, feature_name in enumerate(feature_names):
    feature_vector = X[:, i]
    result = hw3code.find_best_split(feature_vector, y)
    if result is not None:
        thresholds, ginis, best_threshold, best_gini = result
        plt.plot(thresholds, ginis, label=f'{feature_name}', linewidth=2)

plt.xlabel('Порог')
plt.ylabel('Значение критерия Джини')
plt.title('Кривые "порог — значение критерия Джини" для всех признаков')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, (ax, feature_name) in enumerate(zip(axes, feature_names)):
    ax.scatter(X[:, i], y, alpha=0.5, s=20)
    ax.set_xlabel(feature_name)
    ax.set_ylabel('Класс')
    ax.set_title(f'{feature_name} vs Класс')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

__9. (1 балл)__

Исходя из кривых значений критерия Джини, по какому признаку нужно производить деление выборки на два поддерева? Согласуется ли этот результат с визуальной оценкой scatter-графиков? Как бы охарактеризовали вид кривой для "хороших" признаков, по которым выборка делится почти идеально? Чем отличаются кривые для признаков, по которым деление практически невозможно?

**Ответ:**

Исходя из кривых значений критерия Джини, деление выборки на два поддерева нужно производить по признаку с максимальным значением критерия Джини (наивысший пик на графике). Обычно это признак, который лучше всего разделяет классы.

Результат должен согласовываться с визуальной оценкой scatter-графиков: если признак хорошо разделяет классы, то на scatter-графике будет видно четкое разделение точек разных классов по значениям признака.

Для "хороших" признаков, по которым выборка делится почти идеально, кривая Джини имеет высокий и четкий пик, достигающий значений близких к максимуму (около 0.5 для бинарной классификации). Кривая имеет выраженный максимум, что указывает на оптимальный порог разделения.

Для признаков, по которым деление практически невозможно, кривая Джини будет плоской или с очень низкими значениями, близкими к нулю. Это означает, что независимо от выбранного порога, разделение классов будет плохим, и критерий Джини не улучшается значительно.

__10. (1 балл)__

Протестируйте свое решающее дерево на датасете [mushrooms](https://archive.ics.uci.edu/ml/datasets/Mushroom). Вам нужно скачать таблицу agaricus-lepiota.data (из [Data Folder](https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/)), прочитать ее с помощью pandas, применить к каждому столбцу LabelEncoder (из sklearn), чтобы преобразовать строковые имена категорий в натуральные числа. Первый столбец — это целевая переменная (e — edible, p — poisonous) Мы будем измерять качество с помощью accuracy, так что нам не очень важно, что будет классом 1, а что — классом 0. Обучите решающее дерево на половине случайно выбранных объектов (признаки в датасете категориальные) и сделайте предсказания для оставшейся половины. Вычислите accuracy.

У вас должно получиться значение accuracy, равное единице (или очень близкое к единице), и не очень глубокое дерево.

In [ ]:
mushroom_data = pd.read_csv('msh.csv', header=None)

from sklearn.preprocessing import LabelEncoder

encoded_data = mushroom_data.copy()
encoders = {}
for col in encoded_data.columns:
    le = LabelEncoder()
    encoded_data[col] = le.fit_transform(encoded_data[col].astype(str))
    encoders[col] = le

y_mushroom = encoded_data.iloc[:, 0].values
X_mushroom = encoded_data.iloc[:, 1:].values

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_mushroom, y_mushroom, test_size=0.5, random_state=42
)

feature_types = ['categorical'] * X_mushroom.shape[1]
tree = hw3code.DecisionTree(feature_types=feature_types)
tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')

### IIII. Композиции деревьев
#### Задача 4. Сравнение композиционных методов над решающими деревьями
__11. (1 балл)__

Загрузите датасет из соревнования [BNP Paribas Cardif Claims Management](https://www.kaggle.com/c/bnp-paribas-cardif-claims-management/leaderboard). Возьмите из него первые 10к объектов, оставьте только вещественные признаки, а пропуски замените нулями. Разбейте выборку на обучение и контроль в соотношении 7:3.

1. С помощью cross_val_score с cv=3 оцените качество (accuracy) следующих классификаторов на обучающей выборке:
    * DecisionTreeClassifier
    * BaggingClassifier со 100 деревьями
    * RandomForestClassifier со 100 деревьями
    
Значение получается шумное, но в целом у вас должно получиться, что качество возрастает с каждым следующим алгоритмом (если это не так, то посмотрите как ведут себя алгоритмы с разными сидами в кроссвалидации и самих алгоритмах). Этот пример демонстрирует, что RandomForest — это более сложный алгоритм, чем бэггинг.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split

try:
    bnp_data = pd.read_csv('train.csv', nrows=10000)
    
    numeric_cols = bnp_data.select_dtypes(include=[np.number]).columns.tolist()
    if 'target' in numeric_cols:
        numeric_cols.remove('target')
    
    X = bnp_data[numeric_cols].fillna(0).values
    y = bnp_data['target'].values if 'target' in bnp_data.columns else bnp_data.iloc[:, -1].values
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    dt = DecisionTreeClassifier(random_state=42)
    dt_scores = cross_val_score(dt, X_train, y_train, cv=3, scoring='accuracy')
    print(f"DecisionTreeClassifier: {dt_scores.mean():.4f} (+/- {dt_scores.std() * 2:.4f})")
    
    bagging = BaggingClassifier(n_estimators=100, random_state=42)
    bagging_scores = cross_val_score(bagging, X_train, y_train, cv=3, scoring='accuracy')
    print(f"BaggingClassifier (100 trees): {bagging_scores.mean():.4f} (+/- {bagging_scores.std() * 2:.4f})")
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_scores = cross_val_score(rf, X_train, y_train, cv=3, scoring='accuracy')
    print(f"RandomForestClassifier (100 trees): {rf_scores.mean():.4f} (+/- {rf_scores.std() * 2:.4f})")
    
except FileNotFoundError:
    pass

#### Задача 5. Число деревьев в случайном лесе
В этой задаче мы рассмотрим, переобучаются ли композиционные алгоритмы с увеличением числа деревьев.

__12. (1 балл)__

Переберите значения от 20 до 1000-5000 деревьев с шагом 20, посчитайте accuracy на тестовой выборке для каждого числа деревьев и постройте график зависимости качества от числа деревьев.

Рекомендация.

Если каждый раз обучать RandomForest с нуля, придётся обучить в общей сумме $20 + 200 + \ldots + 5000$ деревьев.
Однако, как мы знаем, деревья в случайных лесах строятся независимо и параллельно, поэтому можно обучить всего 5000 деревьев.

Для этого в при создании объекта класса RandomForestClassifier нужно указать в том числе warm_start=True. Затем обучить алгоритм с помощью метода fit, использовать метод predict для классификации. После этого с помощью метода set_params изменить параметр n_estimators. Если к полученному объекту применить метод fit, внутри него будет обучаться только недостающее число деревьев.

Переобучается ли случайный лес с увеличением числа деревьев?

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

try:
    if 'X_train' not in locals() or 'X_test' not in locals():
        bnp_data = pd.read_csv('train.csv', nrows=10000)
        numeric_cols = bnp_data.select_dtypes(include=[np.number]).columns.tolist()
        if 'target' in numeric_cols:
            numeric_cols.remove('target')
        X = bnp_data[numeric_cols].fillna(0).values
        y = bnp_data['target'].values if 'target' in bnp_data.columns else bnp_data.iloc[:, -1].values
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    rf = RandomForestClassifier(n_estimators=20, warm_start=True, random_state=42)
    
    n_trees_list = []
    accuracies = []
    
    max_trees = 1000
    
    for n_trees in range(20, max_trees + 1, 20):
        rf.set_params(n_estimators=n_trees)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        
        n_trees_list.append(n_trees)
        accuracies.append(acc)
    
    plt.figure(figsize=(12, 6))
    plt.plot(n_trees_list, accuracies, 'b-', linewidth=2)
    plt.xlabel('Число деревьев')
    plt.ylabel('Accuracy на тестовой выборке')
    plt.title('Зависимость качества от числа деревьев в RandomForest')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f'Максимальная accuracy: {max(accuracies):.4f} при {n_trees_list[np.argmax(accuracies)]} деревьях')
    
except (FileNotFoundError, NameError):
    pass